In [83]:
def return_lowest_indexed_neighbor(neighborhood_set:set) -> int:
    lowest_index_neighbor = sorted(neighborhood_set)[0]
    return lowest_index_neighbor

def extract_elementary_cycle_from_path(path:list[int]) -> tuple[list[int], list[int]]:
    e_cycle = path[path.index(path[-1]):]
    not_e_cycle = path[:path.index(path[-1])+1]
    return e_cycle, not_e_cycle

def add_edge_back(G:dict[int, set[int]], v1:int, v2:int):
    G[v1].add(v2)
    G[v2].add(v1)

def find_e_cycle_from_edge(e_cycles:list[list[int]], v1:int, v2:int) -> list[int]:
    new_e_cycle = []
    for e_cycle in e_cycles:
        for i in range(len(e_cycle) - 1):
            if e_cycle[i] == v1 and e_cycle[i+1] == v2:
                new_e_cycle = e_cycle[e_cycle.index(v1) + 1 :-1] + e_cycle[0:e_cycle.index(v1)] + [v1]
                break
        if new_e_cycle:
            e_cycles.remove(e_cycle)
            break
    return new_e_cycle

In [89]:
def recursively_find_elementary_cycles(G:dict, path:list[int]=[], elementary_cycles:list[list[int]]=[]) -> list[list[int]]:
    if len(path) != len(set(path)):
        e_cycle, not_e_cycle = extract_elementary_cycle_from_path(path)
        elementary_cycles.append(e_cycle)
        if len(not_e_cycle) > 1:
            for i in range(len(not_e_cycle) - 1):
                add_edge_back(G, not_e_cycle[i], not_e_cycle[i + 1])
        
        return recursively_find_elementary_cycles(G,path=[], elementary_cycles=elementary_cycles)
    
    empty = True
    
    if not path:
        for key, values in G.items():
            if values:
                path.append(key)
                empty = False
                break
        if empty:
            return elementary_cycles
    
    current_vertex = path[-1]
    next_vertex = return_lowest_indexed_neighbor(G[current_vertex])
    G[current_vertex] = G[current_vertex].difference({next_vertex})
    G[next_vertex] = G[next_vertex].difference({current_vertex})
    path.append(next_vertex)
    return recursively_find_elementary_cycles(G, path=path, elementary_cycles=elementary_cycles)

In [112]:
def recursively_construct_euler_circuit(G, e_cycles, stack = [], euler_circuit = []) -> list[int]:
    if not stack:
        if not e_cycles: 
            return euler_circuit
        else:
            stack.append(e_cycles.pop(0))

    v1 = stack[-1].pop(0)
    euler_circuit.append(v1)

    if len(G[v1]) > 2:
        for v2 in G[v1]:
            new_e_cycle = find_e_cycle_from_edge(e_cycles, v1, v2)
            if new_e_cycle:
                stack.append(new_e_cycle)
                break
    if not stack[-1]:
        stack.pop()
    
    return recursively_construct_euler_circuit(G, e_cycles, stack=stack, euler_circuit=euler_circuit)

In [ ]:
def find_euler_circuit(G:dict[int, set]) -> list[int]:
    import copy
    graph_copy = copy.deepcopy(G)
    euler_circuit = []
    e_cycles = recursively_find_elementary_cycles(G=graph_copy, path=[], elementary_cycles=[])
    euler_circuit = recursively_construct_euler_circuit(G=G, e_cycles=e_cycles, stack = [], euler_circuit = [])

    return euler_circuit

[2, 3, 7, 6, 8, 0, 2, 1, 5, 6, 4, 2]

In [131]:
graphs = [
    {
        0: {1,3},
        1: {0,2},
        2: {1,4},
        3: {0,4},
        4: {2,3},
    },
    {
        0: {1, 2, 3, 4},
        1: {0, 2, 3, 4},
        2: {0, 1, 3, 4},
        3: {0, 1, 2, 4},
        4: {0, 1, 2, 3}
    },
    {
        0: {1, 4},
        1: {0, 2},
        2: {1, 3},
        3: {2, 4},
        4: {3, 0}
    },
    {
        0: {1,2},
        1: {0,3},
        2: {0,3},
        3: {1,2,4,5},
        4: {3,6},
        5: {3,6},
        6: {4,5}
    },
    {
        0: {1, 2},
        1: {0, 2, 3, 4},
        2: {0, 1, 3, 5},
        3: {1, 2, 4, 5},
        4: {1, 3, 5, 6},
        5: {2, 3, 4, 6},
        6: {4, 5}
    },
    {
        0: {2, 8},
        1: {2, 5},
        2: {0, 1, 3, 4},
        3: {2, 7},
        4: {2, 6},
        5: {1, 6},
        6: {4, 5, 7, 8},
        7: {3, 6},
        8: {0, 6}
    }
]

for G in graphs:
    print(find_euler_circuit(G))

[0, 1, 2, 4, 3, 0]
[0, 3, 4, 2, 3, 1, 4, 0, 1, 2, 0]
[0, 1, 2, 3, 4, 0]
[0, 1, 3, 4, 6, 5, 3, 2, 0]
[0, 1, 3, 2, 5, 6, 4, 5, 3, 4, 1, 2, 0]
[2, 3, 7, 6, 8, 0, 2, 1, 5, 6, 4, 2]
